# 🔧 Notebook 02: 数据清洗深入 — 15 分钟级电力数据

## Deep Dive into Data Cleaning for 15-Min Power Data

## 学习目标 (Learning Objectives)
1. 理解 15 分钟级电力数据特有的脏数据问题
2. 掌握时序数据缺失值的处理策略
3. 深入理解 IQR 异常值检测及其在电力领域的局限性
4. 学会评估数据质量

## 背景：GIGO 原则 (Garbage In, Garbage Out)

> **Garbage In, Garbage Out** — 垃圾进，垃圾出

如果输入模型的数据是脏的，无论模型多复杂，输出一定是脏的。
数据清洗是机器学习管道中最枯燥但**最重要**的环节。

### 15 分钟级数据特有的脏数据问题

| 问题 | 原因 | 15min 数据中的表现 |
|------|------|-------------------|
| 缺失值 | 传感器故障、网络中断 | 某个 15min 时段没有读数 |
| 异常值 | 传感器故障、极端事件 | 负荷突然跳到 10× 正常值 |
| 价格空值 | 日前市场未出清 | `da_price` 约 75% null (正常) |
| 时间戳跳变 | 采集系统 bug | 00:00 → 00:30 (少了 00:15) |
| 重复点 | 采集重试 | 同一个 15min 出现两次 |
| 时区混乱 | 多源数据 | 北京时间 vs UTC

In [ ]:
from ellectric.pipeline.data_loader import create_loader
from ellectric.pipeline.cleaner import clean_data, validate_schema, get_data_quality_score
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'notebook'

---
## 步骤 1: 加载原始数据，检查"脏"在哪里

In [ ]:
loader = create_loader('shandong')
df_raw = loader.load_data('2024-06-01', '2024-09-30')

print(f"原始数据: {df_raw.shape[0]} 行 × {df_raw.shape[1]} 列")
print(f"时间范围: {df_raw['timestamp'].min()} ~ {df_raw['timestamp'].max()}")

# ── 缺失值全景 ──
print(f"\n═══ 缺失值统计 ═══")
missing = df_raw.isna().sum()
missing_pct = (missing / len(df_raw) * 100)
for col in df_raw.columns:
    n = missing[col]
    if n > 0:
        print(f"  {col:25s}: {n:6d} ({missing_pct[col]:5.1f}%)")

# ── 时间戳连续性检查 ──
print(f"\n═══ 时间戳连续性 ═══")
diffs = df_raw['timestamp'].diff().dropna()
expected = pd.Timedelta(minutes=15)
gaps = diffs[diffs != expected]
print(f"期望间隔: 15 分钟")
print(f"实际间隔: {diffs.min()} ~ {diffs.max()}")
if len(gaps) > 0:
    print(f"⚠ 发现 {len(gaps)} 个时间间隔异常点:")
    print(gaps.value_counts().head())
else:
    print("✅ 时间戳全部连续, 无异常间隔")

### 关键发现: da_price 高缺失率是正常的！

`da_price` (日前电价) 约 75% 缺失是**预期行为**，不是数据质量问题:
- 日前市场每天仅出清次日 24 小时的价格（每小时一个价）
- 15 分钟粒度下，每个小时的 4 个 15-min 点中只有 1 个有日前价格
- 所以 75% (3/4) 的缺失是设计如此

`rt_price` (实时电价) 缺失则要认真对待——它应该是完整的。

---
## 步骤 2: IQR 异常值检测 (只报告不删除)

### IQR (Interquartile Range, 四分位距) 方法

IQR 是最经典的异常值检测方法之一，由统计学家 John Tukey 提出。

**计算步骤:**
1. 将所有数据从小到大排序
2. 找到 Q1 (第 25 百分位) 和 Q3 (第 75 百分位)
3. IQR = Q3 - Q1
4. 正常范围 = [Q1 - 1.5×IQR, Q3 + 1.5×IQR]
5. 超出范围的 → 标记为统计异常值

**为什么是 1.5？**
Tukey 的经验常数。对于正态分布，约 0.7% 的数据落在此范围外。
取 3 过于保守（仅 0.00006%），取 1 过于激进。

In [ ]:
# ── 手动计算 IQR 以理解过程 ────────────────────
q1 = df_raw['load_mw'].quantile(0.25)
q3 = df_raw['load_mw'].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print(f"Q1 (25%): {q1:,.0f} MW")
print(f"Q3 (75%): {q3:,.0f} MW")
print(f"IQR:     {iqr:,.0f} MW")
print(f"正常范围: [{lower:,.0f}, {upper:,.0f}] MW")

outliers = df_raw[(df_raw['load_mw'] < lower) | (df_raw['load_mw'] > upper)]
print(f"\n统计异常值: {len(outliers)} 个 ({len(outliers)/len(df_raw)*100:.2f}%)")

if len(outliers) > 0:
    print(f"\n异常值示例 (前 10):")
    display(outliers[['timestamp', 'load_mw', 'is_holiday', 'is_weekend']].head(10))

# ── 可视化: 分布 + IQR 边界 ──────────────────────
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('负荷分布 + IQR 边界', '负荷时序图 + 异常标记'),
    column_widths=[0.35, 0.65]
)

# 直方图
fig.add_trace(
    go.Histogram(x=df_raw['load_mw'], nbinsx=60, name='负荷分布',
                 marker_color='#1f77b4', showlegend=False),
    row=1, col=1
)
fig.add_vline(x=lower, line_dash='dash', line_color='red', row=1, col=1,
              annotation_text='下限')
fig.add_vline(x=upper, line_dash='dash', line_color='red', row=1, col=1,
              annotation_text='上限')

# 时序图 with outliers
fig.add_trace(
    go.Scatter(x=df_raw['timestamp'], y=df_raw['load_mw'],
               mode='lines', name='负荷', line=dict(color='#1f77b4', width=0.5)),
    row=1, col=2
)
if len(outliers) > 0:
    fig.add_trace(
        go.Scatter(x=outliers['timestamp'], y=outliers['load_mw'],
                   mode='markers', name='IQR 异常值',
                   marker=dict(color='red', size=4, symbol='x')),
        row=1, col=2
    )

fig.update_layout(title='IQR 异常值检测', hovermode='x unified', height=400)
fig.update_xaxes(title_text='负荷 (MW)', row=1, col=1)
fig.update_xaxes(title_text='时间', row=1, col=2)
fig.update_yaxes(title_text='频次', row=1, col=1)
fig.update_yaxes(title_text='负荷 (MW)', row=1, col=2)
fig.show()

### ⚠️ 关键决策: 为什么不删除这些"异常值"？

在电力领域，统计上的"异常值"往往是**真实的极端事件**:

- **夏季极端高温** → 空调全开 → 负荷飙升 (真实的峰值)
- **寒潮来袭** → 取暖负荷激增 (真实的峰值)
- **节假日** → 工业负荷断崖下跌 (真实的谷值)
- **风光满发** → 净负荷骤降 (真实的谷值)

**删除这些数据 = 删除最重要的信号！**

这就是 PITFALLS.md 中的 "spike-as-noise" 反模式:
> 别把尖峰当噪声——尖峰就是信号。

XGBoost 和其他树模型天然善于处理极值（不像线性回归那样被异常值拖偏），
所以保留这些极值 = 保留模型在极端场景下的泛化能力。

IQR 检测的作用是**告警 (Alert)** 而非**清除 (Remove)**——
让你知道数据中哪些时段是异常的（可能有特殊事件），
但把这些异常留给模型自己去学习。

---
## 步骤 3: 执行清洗管道 (Execute Cleaning Pipeline)

`clean_data()` 执行以下操作:
1. ✅ 验证必需列存在 → 列缺失直接报错
2. 🔧 缺失值处理 → 前向填充 (ffill) + 后向填充 (bfill)
3. 📊 IQR 异常值检测 → 记录日志但不删除
4. 🕐 时区标准化 → 统一 UTC

In [ ]:
# ── 执行清洗 ────────────────────────────────────
df_clean = clean_data(df_raw)

# 验证 Schema
result = validate_schema(df_clean)
if result['valid']:
    print("✅ Schema 验证通过")
else:
    print("❌ Schema 验证失败:")
    for issue in result['issues']:
        print(f"  - {issue}")

# ── 清洗前后对比 ──────────────────────────────────
print(f"\n═══ 清洗前后对比 ═══")
print(f"  行数:     {len(df_raw)} → {len(df_clean)}")
print(f"  load_mw 缺失:  {df_raw['load_mw'].isna().sum()} → {df_clean['load_mw'].isna().sum()}")
print(f"  rt_price 缺失: {df_raw['rt_price'].isna().sum()} → {df_clean['rt_price'].isna().sum()}")
print(f"  时区:    {df_raw['timestamp'].dtype} → {df_clean['timestamp'].dtype}")

# ── 缺失值填充效果 ────────────────────────────────
before_missing = df_raw['load_mw'].isna().sum()
before_rt = df_raw['rt_price'].isna().sum()
after_missing = df_clean['load_mw'].isna().sum()
after_rt = df_clean['rt_price'].isna().sum()
print(f"\n  load_mw: 填充了 {before_missing - after_missing} 个空值")
print(f"  rt_price: 填充了 {before_rt - after_rt} 个空值 (ffill 价格是合理的——电力价格有强连续性)")

---
## 步骤 4: 数据质量评分 (Data Quality Score)

In [ ]:
# ── 计算数据质量评分 ────────────────────────────
quality = get_data_quality_score(df_clean)
print(f"═══ 数据质量报告 ═══")
print(f"总分: {quality['quality_score']}/100")
print(f"\n评分明细:")
for dim, val in quality['details'].items():
    print(f"  {dim}: {val}")

---
## 📝 学习笔记 (Key Takeaways)

- **缺失值 ≠ 脏数据** — `da_price` 75% 缺失是设计如此，不是 bug
- **IQR 检测用来告警，不是用来删除** — 电力尖峰是信号，不是噪声
- **ffill 是时序缺失值的最佳默认方案** — 电力负荷有强时间连续性
- **15 分钟粒度让异常检测更精细** — 可以定位到具体 15 分钟时段
- **数据质量评分** 让你量化数据健康状况

## 🔬 深入理解: ffill vs bfill 的选择

前向填充 (ffill): 用上一时刻的值填补空缺
```
时刻:      00:00  00:15  00:30  00:45  01:00
负荷:      45000  NaN    NaN    45200  45500
ffill 后:  45000  45000  45000  45200  45500
```

在电力领域，ffill 优于均值填充，因为:
1. 电力负荷变化是**连续的**——相邻 15 分钟的负荷差通常很小
2. 上午 8 点和凌晨 3 点的负荷完全不是同一个"均值"
3. 上一时刻的负荷是当前时刻的最好估计（持续法思想的延伸）

**下一步: Notebook 03 → 特征工程**

### 思考题 (Reflection Questions)

1. 如果你发现某个 15min 时段 `rt_price = 0`，是应该删除还是保留？为什么？
2. `pumped_storage_mw` (抽水蓄能) 有时为正值 (抽水) 有时为负值 (发电)。IQR 方法能处理有正有负的数据吗？
3. 如果 15 分钟数据的缺失率是 5% (每天漏 5 个点)，前向填充会引入多大误差？